# Credit Explainer — Phase 1 ③ 생성형 AI 계층: 자연어 설명문 생성

`01_demo_core.ipynb`에서 산출한 **SHAP Local Top 5 + Counterfactual 결과**를 입력으로,
소비자 눈높이의 자연어 설명문을 생성한다. **평가와 설명의 역할 분리** 원칙에 따라
LLM은 평가에 관여하지 않고 이미 산출된 근거를 "번역"만 한다 (ADR-0001).

### 사용 모델·파라미터 기록 (ADR-0001 4항)

| 항목 | 값 |
|---|---|
| 모델 | **`claude-opus-4-8`** (구현 시점 2026-07 Anthropic 권장 기본 모델, 코드 상수로 고정) |
| max_tokens | 800 |
| system prompt | 프롬프트 5요소 공식(역할·맥락·목표·형식·제약) — `src/explainer.py` |
| sampling | 미지정 (claude-opus-4-8은 temperature 등 샘플링 파라미터를 받지 않음) |
| 요금 | 입력 $5 / 출력 $25 per 1M tokens |
| 안전장치 | 세션 호출 12회 한도, 비용 상한 $5, 캐시 우선(`outputs/explanations_cache.json`) |

### 발표장 모드 (ADR-0001 2항)

`USE_CACHE=True`(기본)일 때 캐시가 있으면 **API 호출 0회**로 설명문을 재생한다.
발표장 네트워크가 없어도 이 노트북은 끝까지 실행된다.

> **수치 표기 원칙** — 성능 수치는 01 노트북 실행값(홀드아웃 **AUC 0.9230 / KS 0.7364**)으로
> 통일해 인용한다. 사례별 PD도 01 실행값을 그대로 사용한다.

## 0. 사전 확인 — 금리 컬럼 단위 검증

01 노트북에서 경계 사례의 `최고금리`가 **22,900**으로 표시되었다. 설명문에 수치를 그대로
인용하면 소비자가 읽을 수 없으므로, 원본 데이터의 금리 3개 컬럼 분포로 실제 스케일을 판정한다.

**판정: 금리는 `% × 1000` 스케일로 저장되어 있다 (22,900 = 연 22.9%).** 근거:

1. 세 금리 컬럼 모두 0을 제외한 **최댓값이 24,000(≈23,900)** — **2018.2~2021.7 적용
   법정 최고금리 24%와 일치**한다 (2021.7.7부터는 20%로 인하, 출처: 금융위 보도자료).
   우연히 일치하기 어려운 규제 상한이며, 데이터 수집 시기가 해당 구간임을 시사한다.
2. ÷1000 변환 시 분포가 현실적 금리 대역에 놓인다: 신용대출 중앙값 5.0%,
   장기카드대출(카드론) 14.9%, 단기카드대출(현금서비스) 18.45% — 상품별 통상 금리 순서와 일치.
3. 파생변수 `잔액가중평균금리` 19,190.8 → 19.19%로 일관된다.

이에 따라 `src/explainer.py`의 `display_value()`가 금리를 `연 22.9%` 형태로 변환해
LLM 입력에 사용한다. **잔액류 컬럼은 데이터 정의서가 없어 화폐 단위를 확정할 수 없으므로**
단위를 붙이지 않고 수치와 상대 변화(감소율)로만 표현한다.

In [1]:
import warnings, sys, os, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd

df_raw = pd.read_csv('../../data/AI_Play_DB.csv', encoding='cp949')
rate_cols = ['신용대출금리', '장기카드대출금리', '단기카드대출금리']
dist = pd.DataFrame({c: df_raw.loc[df_raw[c] > 0, c].describe()[['count', 'min', '50%', 'max']]
                     for c in rate_cols}).T
dist.columns = ['보유자 수', '최솟값', '중앙값', '최댓값']
dist['÷1000 → 중앙값(%)'] = dist['중앙값'] / 1000
dist['÷1000 → 최댓값(%)'] = dist['최댓값'] / 1000
display(dist)
print('세 컬럼 모두 최댓값 ÷1000 = 24% 부근(2018.2~2021.7 법정 최고금리) → % × 1000 스케일로 판정')

,보유자 수,최솟값,중앙값,최댓값,÷1000 → 중앙값(%),÷1000 → 최댓값(%)
신용대출금리,23973.0,1000.0,5000.0,24000.0,5.00,24.0
장기카드대출금리,6822.0,2025.0,14900.0,23900.0,14.90,23.9
단기카드대출금리,5594.0,1000.0,18450.0,24000.0,18.45,24.0


세 컬럼 모두 최댓값 ÷1000 = 24% 부근(2018.2~2021.7 법정 최고금리) → % × 1000 스케일로 판정


## 1. 입력 로드 — 01 노트북 산출물

- `outputs/shap_top5_cases.csv`: 사례 3건 × 상위 5개 기여 변수 (SHAP log-odds)
- `outputs/counterfactual_result.json`: 경계 사례의 최소 변화 조합
- 사례 PD·판정은 01 노트북 실행값 인용: 사례 1 **50.12% 거절**, 사례 2 **0.00% 승인**,
  사례 3 **4.49% 거절**(임계값 4.43%)

In [2]:
from explainer import (MODEL, display_value, build_prompt, generate_explanation,
                       SYSTEM_PROMPT, CACHE_PATH)
import explainer

# API 키 존재 확인 (값은 출력하지 않음)
from dotenv import load_dotenv
load_dotenv('../.env')
print('ANTHROPIC_API_KEY 설정됨:', bool(os.getenv('ANTHROPIC_API_KEY')))
print('사용 모델:', MODEL)

shap_top5 = pd.read_csv('../outputs/shap_top5_cases.csv', index_col=[0, 1])
cf = json.load(open('../outputs/counterfactual_result.json', encoding='utf-8'))

case_meta = {  # 01 노트북 실행값 인용
    1: {'사례명': '고위험 거절', '판정': '거절', 'pd': 0.5012},
    2: {'사례명': '저위험 승인', '판정': '승인', 'pd': 0.0000},
    3: {'사례명': '경계(임계값 부근)', '판정': '거절', 'pd': cf['current_pd']},
}
THRESHOLD = cf['threshold']

cf_text = (f"단기카드대출잔액을 {cf['best']['단기카드잔액 감소율']:.0%} 상환하는 경우, "
           f"부실 위험 추정치가 {cf['current_pd'] * 100:.2f}%에서 "
           f"{cf['best']['예상 PD'] * 100:.2f}%로 내려가 판정 기준값({THRESHOLD * 100:.2f}%) 미만이 됨")

cases = []
for i in (1, 2, 3):
    rows = shap_top5.loc[i]
    factors = [{'변수': r['변수'], '표시값': display_value(r['변수'], r['고객값']),
                '방향': '증가' if r['SHAP(log-odds)'] > 0 else '완화'}
               for _, r in rows.iterrows()]
    cases.append({'case_id': f'case_{i}', '사례명': case_meta[i]['사례명'],
                  '판정': case_meta[i]['판정'], 'pd': case_meta[i]['pd'],
                  'threshold': THRESHOLD, 'factors': factors,
                  'counterfactual': cf_text if i == 3 else None})

print('\n=== 사례 3(경계) LLM 입력 예시 ===')
print(build_prompt(cases[2]))

ANTHROPIC_API_KEY 설정됨: True
사용 모델: claude-opus-4-8

=== 사례 3(경계) LLM 입력 예시 ===
[평가 결과] 거절
[부실 위험 추정치] 4.49% (판정 기준값: 4.43% 이상이면 거절)
[주요 요인 (영향력 순)]
  1. 최고금리 = 연 22.9% — 위험 증가 요인
  2. 잔액가중평균금리 = 연 19.19% — 위험 증가 요인
  3. 단기카드대출금리 = 연 22.9% — 위험 완화 요인
  4. 월평균신용카드사용액 = 436 — 위험 완화 요인
  5. 카드사용액합계 = 626 — 위험 완화 요인
[개선 시뮬레이션 결과] 단기카드대출잔액을 25% 상환하는 경우, 부실 위험 추정치가 4.49%에서 2.06%로 내려가 판정 기준값(4.43%) 미만이 됨

위 정보만 사용하여 형식에 맞는 설명문을 작성해 주세요.


## 2. 설명문 생성 (캐시 우선)

`generate_explanation()`은 캐시에 있으면 그대로 재생하고, 없을 때만 API를 호출한 뒤
`outputs/explanations_cache.json`에 저장한다. 최초 1회 라이브 실행으로 캐시를 만들었고,
이후 실행(발표장 포함)은 모두 캐시 재생이다.

In [3]:
results = {}
for case in cases:
    r = generate_explanation(case, use_cache=True)
    results[case['case_id']] = r
    src = '캐시 재생' if r['from_cache'] else '라이브 API 호출'
    print(f"{'=' * 72}")
    print(f"[{case['case_id']} · {case['사례명']}] {src} | {r['model']} | "
          f"입력 {r['usage']['input_tokens']} / 출력 {r['usage']['output_tokens']} tokens | "
          f"${r['cost_usd']:.4f} | 생성 {r['created_at']}")
    print(f"{'=' * 72}")
    print(r['text'])
    print()
print(f'이번 세션 API 호출: {explainer.api_calls}회 / 세션 비용: ${explainer.total_cost_usd:.4f}')
total = sum(r['usage']['input_tokens'] + r['usage']['output_tokens'] for r in results.values())
total_cost = sum(r['cost_usd'] for r in results.values())
print(f'사례 3건 누적(캐시 생성 시점 기준): 총 {total:,} tokens, ${total_cost:.4f}')

[case_1 · 고위험 거절] 캐시 재생 | claude-opus-4-8 | 입력 1063 / 출력 486 tokens | $0.0175 | 생성 2026-07-27T13:26:18+00:00
① 결과 통지
이번 신용평가 결과, 고객님의 대출 신청은 승인되지 않았습니다(부실 위험 추정치 50.12%로, 거절 기준값인 4.43% 이상에 해당).

② 주요 요인
이 결과에 영향을 준 주요 요인은 다음과 같습니다. 첫째, 연체와 관련된 지표 4개 중 4개 모두에 해당하여 위험을 높였습니다. 둘째, 제3금융권(대부업 등) 잔액 비중이 94.9%로 높았습니다. 셋째, 제1금융권(은행)이 아닌 곳의 잔액 비중이 100.0%였습니다. 넷째, 신용카드를 개설한 기관이 2개였고, 다섯째, 신용성 대출 비중이 100.0%인 점이 위험 증가 요인으로 작용했습니다.

③ 개선 참고사항
연체 관련 지표나 비은행권·신용성 대출 비중 등이 개선되면 재평가 시 결과가 달라질 수 있습니다.

④ 권리 안내
고객님께서는 이 평가에 대한 설명을 요구하실 수 있으며, 기초정보를 제출하거나 잘못된 정보의 정정을 요청하실 수 있습니다. 또한 결과에 대해 이의를 제기하실 수 있습니다.

[case_2 · 저위험 승인] 캐시 재생 | claude-opus-4-8 | 입력 1039 / 출력 494 tokens | $0.0175 | 생성 2026-07-27T13:26:26+00:00
① 결과 통지
고객님의 신용평가 결과는 **승인**으로 판정되었습니다.

② 주요 요인
이번 평가에서 추정된 부실 위험은 0.00%로, 거절 기준값인 4.43%보다 낮았습니다. 결과에 긍정적으로 작용한 주요 요인은 다음과 같습니다. 첫째, 전체 사용 중 신용카드 사용 비중이 100.0%로 나타났습니다. 둘째, 신용카드를 개설하신 기관 수가 3개였습니다. 셋째, 보험료 합계가 12로 확인되었습니다. 넷째, 월평균 신용카드 사용액이 1,056이었으며, 다섯째, 연금보험료가 4로 나타났습니다. 이 다섯 항목이 모두 위험을 낮추는 

## 3. 발표장 모드 검증 — 캐시 재생 시 API 호출 0회 (ADR-0001 2항)

In [4]:
USE_CACHE = True  # 발표장 기본 모드

calls_before = explainer.api_calls
replay = {c['case_id']: generate_explanation(c, use_cache=USE_CACHE) for c in cases}

assert all(r['from_cache'] for r in replay.values()), '캐시 미사용 사례 발견'
assert explainer.api_calls == calls_before, 'API가 호출됨'
assert CACHE_PATH.exists(), '캐시 파일 없음'
print(f'✔ 캐시 재생 확인 — API 호출 {explainer.api_calls - calls_before}회, '
      f'설명문 {len(replay)}건 모두 {CACHE_PATH.name}에서 재생 (네트워크 불필요)')

✔ 캐시 재생 확인 — API 호출 0회, 설명문 3건 모두 explanations_cache.json에서 재생 (네트워크 불필요)


## 4. HITL 검토 (Human-in-the-Loop) — GDPR 22조 인적 개입 대응

생성된 설명문 3건을 사람이 검토한 결과. 검토 기준: (a) **수치 일치** — 설명문의 모든 수치가
LLM 입력(표시값)과 일치하는가, (b) **금지 문구 없음** — 점수 조작 유도·확정적 승인 약속·투자
권유·입력 외 수치 생성이 없는가, (c) **비전문가 이해 용이성** — 전문용어를 풀어 썼는가.

| 사례 | 수치 일치 | 금지 문구 없음 | 이해 용이성 | 판정 |
|---|---|---|---|---|
| 1 (고위험 거절) | ✅ 50.12% · 4.43% · 4개/4개 · 94.9% · 100.0% · 2개 모두 입력과 일치 | ✅ "개선되면 ~ 달라질 수 있습니다" 비확정 표현만 사용 | ✅ 3금융권을 "대부업 등"으로 풀어 설명 | **통과** |
| 2 (저위험 승인) | ✅ 0.00% · 4.43% · 100.0% · 3개 · 12 · 1,056 · 4 모두 일치 | ✅ 승인 사례임에도 향후 승인 보장 표현 없음, CF 부재를 정직하게 안내 | ✅ 요인을 순서대로 짧은 문장으로 나열 | **통과** |
| 3 (경계 거절) | ✅ 4.49% · 4.43% · 연 22.9% · 연 19.19% · 436 · 626 · 25% · 2.06% 모두 일치 | ✅ CF 결과를 "재평가 시 결과가 달라질 수 있습니다"로만 안내 (확정 약속 없음) | ✅ 잔액가중평균금리에 괄호 부연 설명 추가 | **통과** |

**검토자 코멘트 (개선 여지, 통과에는 영향 없음)**

1. 사례 2의 "보험료 합계가 12" — 잔액류 화폐 단위 미확정 방침(0절)에 따른 의도된 출력이나,
   소비자 입장에서는 어색하다. 실서비스 전 데이터 정의서 확보 후 단위 표기가 필요하다.
2. 사례 3에서 `최고금리`(위험 증가)와 `단기카드대출금리`(위험 완화)가 같은 값(연 22.9%)으로
   상반된 방향으로 서술됨 — SHAP 입력에 충실한 결과이지만 소비자 혼란 소지가 있어,
   XAI 단계에서 상관 높은 변수의 그룹핑(예: 금리 요인 통합)을 향후 개선점으로 기록한다.
3. 분량 369~415자(공백 제외) — "400자 내외" 형식 지시 준수.

**종합: 3건 모두 통과 — 환각(입력 외 수치·사실 생성) 0건.**\


## 5. 규제 매핑 — 파이프라인 구성요소 ↔ 규제 근거

| 파이프라인 구성요소 | 신용정보법 제36조의2 (설명요구권) | EU GDPR 제22조 (자동화 결정) | 금융분야 AI 가이드라인 (7대 원칙 관련) |
|---|---|---|---|
| ① 판단형 AI (EasyEnsemble 20×HGB + Platt) | 자동화평가 '결과'의 산출 주체 — 설명 대상이 되는 평가 | 프로파일링을 포함한 자동화 결정 | 안전성·건전성: 검증된 모형 재사용, 임계값 문서화 |
| ② XAI (SHAP Local · Counterfactual) | '주요 기준'(상위 요인)과 '기초정보' 산출 → 설명요구권의 실질적 근거 | 결정 논리에 관한 유의미한 정보 제공 | 투명성·설명가능성: 개인 단위 근거 산출 |
| ③ 생성형 AI (Claude, 번역만) | 소비자가 이해할 수 있는 형식의 설명 제공 + **정보제출권·이의제기권 안내 문구 포함** | 이해 가능한 방식의 정보 제공 | 소비자 권익 보호: 비전문가 눈높이 설명 |
| ④-a HITL 검토 (본 노트북 4절) | 설명 내용의 정확성 검증 | **인적 개입을 요구할 권리** (제22조 3항) 대응 | 책임성: 사람의 최종 검토·개입 지점 확보 |
| ④-b 공정성 감사 (01 노트북 6절) | — | — | 공정성: 프록시 그룹 세분화 평가 (DP·EO) |
| 캐시·템플릿 제약 (ADR-0001) | 설명 내용의 일관성 보장 | — | 안전성: 환각·조작 리스크 통제 (입력 외 수치 생성 금지) |

> 역할 분리 원칙: 판단형 AI는 평가만, XAI는 근거 산출만, 생성형 AI는 번역만.
> LLM이 평가에 관여하지 않으므로 "생성형 AI가 신용점수를 좌우한다"는 리스크가 구조적으로 차단된다.

## 6. 결론

- 실습과제 모형(홀드아웃 AUC 0.9230 / KS 0.7364, 01 노트북 재현값) 위에
  ①판단 → ②근거(SHAP·CF) → ③번역(LLM) → ④통제(HITL·공정성 감사)의
  **신뢰 계층 전 과정이 작동함**을 확인했다.
- 설명문 3건은 `outputs/explanations_cache.json`에 저장되어 발표장에서
  **네트워크 없이 재생**된다 (3절 검증).
- **한계**: LLM 환각 리스크는 템플릿 제약 + HITL로 완화하되 잔존(ADR-0001).
  표시값의 화폐 단위(잔액류)는 데이터 정의서 부재로 미확정이라 단위 없이 인용했다.
- **다음 단계**: Phase 2 발표자료 — 01·02 노트북의 차트·설명문 화면을 10분 구성안에 조립.